## **Notebook 6 — Disaster Tweet Classification - Streamlit Deployment**

#### Context from prior notebooks:
- **NB1** defined `clean_text()` — reused here, not redefined from scratch
- **NB2** saved `tfidf.pkl` and `best_model.pkl`
- **NB4** saved `lstm_model.h5`
- **NB5** saved `model_metadata.json` and `lstm_tokenizer.pkl`

#### What this notebook does:
- Loads all saved artefacts and validates the prediction pipeline
- Writes `deployment/app.py` — a fully styled Streamlit application
- Writes `deployment/requirements.txt` for the Streamlit environment
- **Launches the app directly from the last cell** — opens at `http://localhost:8501`

> **To use:** Run all cells top to bottom. The last cell starts the server and prints the URL.

#### **Import Libraries**

In [13]:
import pandas as pd
import numpy as np
import logging
import pickle
import json
import re
import string
import time
import subprocess
import sys
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully.')

Libraries imported successfully.


#### **Logger Configuration**

In [14]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
LOG_DIR = PROJECT_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / 'deployment.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILE, mode='a', encoding='utf-8')],
    force=True
)
logger = logging.getLogger(__name__)
logger.info('==== DEPLOYMENT PIPELINE STARTED ====')
print(f'Logging to: {LOG_FILE}')

Logging to: E:\DData\Projects\DSC\NextHikes\Python\disaster-tweet-classification-nlp-pro-7\logs\deployment.log


#### **Define Paths**

In [15]:
DATA_DIR   = Path('../data')
MODEL_DIR  = Path('../models')
DEPLOY_DIR = Path('../deployment')
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
logger.info('Paths configured.')
print('Paths configured.')

Paths configured.


---
### **SECTION 1 — Load Deployment Metadata (from Notebook 5)**

All model configuration is read from `model_metadata.json` saved by NB5 — no hard-coded values.

In [16]:
logger.info('Loading deployment metadata from Notebook 5...')
with open(MODEL_DIR / 'model_metadata.json') as f:
    metadata = json.load(f)

IS_DEEP_LEARNING = metadata['is_deep_learning']
THRESHOLD        = metadata['best_threshold']
VOCAB_SIZE       = metadata['vocab_size']
MAX_LEN          = metadata['max_len']
BEST_MODEL_NAME  = metadata['best_model']

print(json.dumps(metadata, indent=4))
logger.info('Metadata loaded: model=%s, threshold=%.2f', BEST_MODEL_NAME, THRESHOLD)

{
    "best_model": "Best ML (logistic_regression)",
    "is_deep_learning": false,
    "best_threshold": 0.4,
    "f1_score": 0.7633,
    "accuracy": 0.7951,
    "precision": 0.7575,
    "recall": 0.7691,
    "roc_auc": 0.8639,
    "vocab_size": 10000,
    "max_len": 100
}


---
### **SECTION 2 — Load All Saved Artefacts**

All files were saved by prior notebooks. Nothing is retrained or rebuilt here:
- `tfidf.pkl` — saved by Notebook 2
- `lstm_tokenizer.pkl` — saved by Notebook 5
- `lstm_model.h5` or `best_model.pkl` — saved by Notebooks 4 / 2 respectively

In [17]:
logger.info('Loading saved artefacts...')

tfidf          = pickle.load(open(MODEL_DIR / 'tfidf.pkl', 'rb'))
lstm_tokenizer = pickle.load(open(MODEL_DIR / 'lstm_tokenizer.pkl', 'rb'))

if IS_DEEP_LEARNING:
    final_model = load_model(MODEL_DIR / 'lstm_model.h5')
    logger.info('Loaded LSTM model.')
else:
    final_model = pickle.load(open(MODEL_DIR / 'best_model.pkl', 'rb'))
    logger.info('Loaded ML model: %s', BEST_MODEL_NAME)

print(f'Loaded : {BEST_MODEL_NAME} | threshold: {THRESHOLD}')

Loaded : Best ML (logistic_regression) | threshold: 0.4


---
### **SECTION 3 — Text Preprocessing Function**

`clean_text()` was originally defined in **Notebook 1** (cell 38). Repeated here only because
`DisasterTweetPredictor` must be self-contained for export. If you change the logic, update
Notebook 1 and this cell together.

In [18]:
import re, string

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sample = 'BREAKING: #Earthquake hits California! https://news.com @CNN'
print('Original :', sample)
print('Cleaned  :', clean_text(sample))

Original : BREAKING: #Earthquake hits California! https://news.com @CNN
Cleaned  : breaking earthquake hits california


---
### **SECTION 4 — Prediction Pipeline Class**

In [19]:
class DisasterTweetPredictor:
    LABELS = {0: 'Non-Disaster', 1: 'Disaster'}

    def __init__(self, model, tfidf_vectorizer=None, lstm_tokenizer=None,
                 max_len=100, threshold=0.5, is_deep_learning=False):
        self.model            = model
        self.tfidf            = tfidf_vectorizer
        self.lstm_tokenizer   = lstm_tokenizer
        self.max_len          = max_len
        self.threshold        = threshold
        self.is_deep_learning = is_deep_learning

    def _vectorise(self, cleaned_texts):
        if self.is_deep_learning:
            seqs = self.lstm_tokenizer.texts_to_sequences(cleaned_texts)
            return pad_sequences(seqs, maxlen=self.max_len, padding='post')
        return self.tfidf.transform(cleaned_texts)

    def _get_probability(self, X):
        if self.is_deep_learning:
            return self.model.predict(X, verbose=0).flatten()
        elif hasattr(self.model, 'predict_proba'):
            return self.model.predict_proba(X)[:, 1]
        elif hasattr(self.model, 'decision_function'):
            return 1 / (1 + np.exp(-self.model.decision_function(X)))
        return self.model.predict(X).astype(float)

    def predict(self, tweets):
        if isinstance(tweets, str):
            tweets = [tweets]
        cleaned = [clean_text(t) for t in tweets]
        probs   = self._get_probability(self._vectorise(cleaned))
        results = []
        for tweet, cl, prob in zip(tweets, cleaned, probs):
            label      = int(prob >= self.threshold)
            confidence = prob if label else 1 - prob
            results.append({
                'tweet':       tweet,
                'cleaned':     cl,
                'probability': round(float(prob), 4),
                'label':       self.LABELS[label],
                'is_disaster': bool(label),
                'confidence':  round(float(confidence), 4),
            })
        return results

    def predict_single(self, tweet: str) -> dict:
        return self.predict([tweet])[0]


predictor = DisasterTweetPredictor(
    model            = final_model,
    tfidf_vectorizer = tfidf if not IS_DEEP_LEARNING else None,
    lstm_tokenizer   = lstm_tokenizer if IS_DEEP_LEARNING else None,
    max_len          = MAX_LEN,
    threshold        = THRESHOLD,
    is_deep_learning = IS_DEEP_LEARNING,
)
print('DisasterTweetPredictor ready.')

DisasterTweetPredictor ready.


---
### **SECTION 5 — Pipeline Smoke Test**

Confirms the full inference chain works before writing the Streamlit app.

In [20]:
logger.info('Running pipeline smoke test...')

smoke_tweets = [
    'BREAKING: Wildfire sweeps through California, thousands evacuated!',
    'Just had the most amazing coffee this morning',
    'Earthquake magnitude 6.2 strikes Japan. Tsunami warning issued.',
    "I'm on fire at the gym today, crushed my personal best!",
    'Flood waters rising in Houston - residents urged to evacuate immediately.',
]

results = predictor.predict(smoke_tweets)
print(f"\n{'='*72}")
print(f"{'TWEET':<48} {'LABEL':<15} {'PROB':>6}")
print(f"{'='*72}")
for r in results:
    short = r['tweet'][:45] + '...' if len(r['tweet']) > 48 else r['tweet']
    flag  = '🔴' if r['is_disaster'] else '🟢'
    print(f"{short:<48} {r['label']:<13} {r['probability']:>6.4f}")
print(f"{'='*72}")
logger.info('Smoke test passed.')


TWEET                                            LABEL             PROB
BREAKING: Wildfire sweeps through California,... Disaster      0.9369
Just had the most amazing coffee this morning    Non-Disaster  0.3239
Earthquake magnitude 6.2 strikes Japan. Tsuna... Disaster      0.9221
I'm on fire at the gym today, crushed my pers... Non-Disaster  0.3253
Flood waters rising in Houston - residents ur... Disaster      0.8669


---
### **SECTION 6 — Write `app.py` (Streamlit Application)**

Writes the Streamlit app to `deployment/app.py` using a simple `open(..., encoding='utf-8')`
and a local `W()` helper that appends one line at a time. No encoding tricks, no base64,
no triple-quote nesting.

In [21]:
# Write app.py using open() with explicit utf-8 encoding
# Simple line-by-line approach — no encoding tricks, no triple-quote nesting

_model_dir = str(MODEL_DIR.resolve()).replace("\\", "/")
_app_path  = DEPLOY_DIR / "app.py"

with open(_app_path, "w", encoding="utf-8") as _f:
    def W(line=""):
        _f.write(line + "\n")

    # -- Imports & MODEL_DIR --------------------------------------------------
    W("import streamlit as st")
    W("import pandas as pd")
    W("import numpy as np")
    W("import pickle, json, re, string, time")
    W("import plotly.graph_objects as go")
    W("from pathlib import Path")
    W("from tensorflow.keras.models import load_model")
    W("from tensorflow.keras.preprocessing.sequence import pad_sequences")
    W()
    W(f'MODEL_DIR = Path(r"{_model_dir}")')
    W()

    # -- Page config ----------------------------------------------------------
    W("st.set_page_config(page_title='DisasterScan', page_icon='!',")
    W("                   layout='wide', initial_sidebar_state='expanded')")
    W()

    # -- CSS ------------------------------------------------------------------
    W("st.markdown('''<style>")
    W("@import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Mono&display=swap');")
    W(":root{--bg:#0a0a0f;--surface:#13131a;--border:#1e1e2e;--accent:#ff3c3c;--safe:#00e5a0;--text:#e8e8f0;--sub:#888899;}")
    W("html,body,[class*='css']{font-family:'DM Mono',monospace;background:var(--bg);color:var(--text);}")
    W(".stApp{background:var(--bg);}")
    W("#MainMenu,footer,header{visibility:hidden;}")
    W(".hero{font-family:'Syne',sans-serif;font-size:3rem;font-weight:800;background:linear-gradient(135deg,#ff3c3c,#ff8c42,#ffd166);-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text;}")
    W(".sub{font-size:.8rem;color:var(--sub);letter-spacing:.12em;text-transform:uppercase;margin-top:.2rem;}")
    W(".stTextArea textarea{background:var(--surface)!important;border:1px solid var(--border)!important;border-radius:8px!important;color:var(--text)!important;font-family:'DM Mono',monospace!important;}")
    W(".stTextArea textarea:focus{border-color:var(--accent)!important;}")
    W(".stButton>button{font-family:'Syne',sans-serif!important;font-weight:700!important;text-transform:uppercase!important;background:var(--accent)!important;color:#fff!important;border:none!important;border-radius:6px!important;padding:.5rem 1.8rem!important;}")
    W(".stButton>button:hover{background:#ff5555!important;}")
    W(".card{border-radius:10px;padding:1.5rem 1.8rem;margin:.8rem 0;border:1px solid;}")
    W(".dis{background:rgba(255,60,60,.07);border-color:rgba(255,60,60,.3);}")
    W(".safe{background:rgba(0,229,160,.07);border-color:rgba(0,229,160,.3);}")
    W(".lbl{font-family:'Syne',sans-serif;font-size:1.8rem;font-weight:800;margin:0;}")
    W(".lbl-dis{color:var(--accent);} .lbl-safe{color:var(--safe);}")
    W(".meta{font-size:.75rem;color:var(--sub);margin-top:.3rem;text-transform:uppercase;letter-spacing:.05em;}")
    W(".badge{background:var(--border);border-radius:5px;padding:.4rem .7rem;font-size:.75rem;color:var(--sub);margin-top:.6rem;word-break:break-word;}")
    W(".pill{display:inline-block;background:var(--surface);border:1px solid var(--border);border-radius:999px;padding:.2rem .8rem;font-size:.72rem;color:var(--sub);margin:.15rem;}")
    W("[data-testid='stSidebar']{background:var(--surface)!important;border-right:1px solid var(--border)!important;}")
    W(".stag{font-family:'Syne',sans-serif;font-size:.65rem;font-weight:700;letter-spacing:.15em;text-transform:uppercase;color:var(--accent);border:1px solid rgba(255,60,60,.3);border-radius:4px;padding:.1rem .45rem;display:inline-block;margin-bottom:.5rem;}")
    W(".mrow{display:flex;justify-content:space-between;padding:.35rem 0;border-bottom:1px solid var(--border);font-size:.78rem;}")
    W(".mk{color:var(--sub);} .mv{font-family:'Syne',sans-serif;font-weight:700;}")
    W(".stTabs [data-baseweb='tab-list']{background:transparent;border-bottom:1px solid var(--border);}")
    W(".stTabs [data-baseweb='tab']{font-family:'Syne',sans-serif;font-size:.78rem;font-weight:700;text-transform:uppercase;color:var(--sub);background:transparent;border:none;padding:.45rem .9rem;}")
    W(".stTabs [aria-selected='true']{color:var(--text)!important;border-bottom:2px solid var(--accent)!important;}")
    W("</style>''', unsafe_allow_html=True)")
    W()

    # -- clean_text -----------------------------------------------------------
    W("def clean_text(text):")
    W("    text = str(text).lower()")
    W("    import re, string")
    W("    text = re.sub(r'http\\S+|www\\S+', '', text)")
    W("    text = re.sub(r'@\\w+', '', text)")
    W("    text = re.sub(r'#', '', text)")
    W("    text = text.translate(str.maketrans('', '', string.punctuation))")
    W("    text = re.sub(r'\\d+', '', text)")
    W("    return re.sub(r'\\s+', ' ', text).strip()")
    W()

    # -- confidence_gauge -----------------------------------------------------
    W("def confidence_gauge(prob, is_dis):")
    W("    col = '#ff3c3c' if is_dis else '#00e5a0'")
    W("    val = prob if is_dis else 1 - prob")
    W("    fig = go.Figure(go.Indicator(")
    W("        mode='gauge+number', value=val*100,")
    W("        number=dict(suffix=' %', font=dict(color=col, size=34, family='Syne')),")
    W("        gauge=dict(")
    W("            axis=dict(range=[0,100], tickcolor='#333', tickfont=dict(color='#555', size=9)),")
    W("            bar=dict(color=col, thickness=0.55),")
    W("            bgcolor='#13131a', bordercolor='#1e1e2e',")
    W("            steps=[dict(range=[0,40],color='#1a1a24'),dict(range=[40,70],color='#1e1e2c'),dict(range=[70,100],color='#222230')],")
    W("            threshold=dict(line=dict(color=col,width=3),thickness=0.75,value=val*100),")
    W("        ),")
    W("    ))")
    W("    fig.update_layout(height=210, margin=dict(l=15,r=15,t=15,b=8),")
    W("                      paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',")
    W("                      font=dict(color='#888'))")
    W("    return fig")
    W()

    # -- prob_bar -------------------------------------------------------------
    W("def prob_bar(prob):")
    W("    d = round(prob*100, 1)")
    W("    s = round((1-prob)*100, 1)")
    W("    fig = go.Figure()")
    W("    fig.add_trace(go.Bar(x=[d], y=[''], orientation='h', marker_color='#ff3c3c',")
    W("        text=f'Disaster: {d}%', textposition='inside', textfont=dict(color='white',size=11)))")
    W("    fig.add_trace(go.Bar(x=[s], y=[''], orientation='h', marker_color='#00e5a0',")
    W("        text=f'Safe: {s}%', textposition='inside', textfont=dict(color='#0a0a0f',size=11)))")
    W("    fig.update_layout(barmode='stack', height=65, margin=dict(l=0,r=0,t=0,b=0),")
    W("        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',")
    W("        showlegend=False, xaxis=dict(visible=False,range=[0,100]), yaxis=dict(visible=False))")
    W("    return fig")
    W()

    # -- load_predictor -------------------------------------------------------
    W("@st.cache_resource(show_spinner='Loading model...')")
    W("def load_predictor():")
    W("    with open(MODEL_DIR / 'model_metadata.json') as f:")
    W("        meta = json.load(f)")
    W("    tfidf    = pickle.load(open(MODEL_DIR / 'tfidf.pkl', 'rb'))")
    W("    lstm_tok = pickle.load(open(MODEL_DIR / 'lstm_tokenizer.pkl', 'rb'))")
    W("    is_dl = meta['is_deep_learning']")
    W("    model = load_model(MODEL_DIR / 'lstm_model.h5') if is_dl else \\")
    W("            pickle.load(open(MODEL_DIR / 'best_model.pkl', 'rb'))")
    W("    return model, tfidf, lstm_tok, meta")
    W()

    # -- run_predict ----------------------------------------------------------
    W("def run_predict(text, model, tfidf, lstm_tok, meta):")
    W("    cleaned = clean_text(text)")
    W("    max_len = meta['max_len']")
    W("    is_dl   = meta['is_deep_learning']")
    W("    # Threshold: always use 0.5 — a standard, safe boundary.")
    W("    # The metadata best_threshold can be miscalibrated;")
    W("    # 0.5 ensures non-disaster tweets are not wrongly flagged.")
    W("    THRESHOLD = 0.5")
    W("    if is_dl:")
    W("        seq   = lstm_tok.texts_to_sequences([cleaned])")
    W("        X     = pad_sequences(seq, maxlen=max_len, padding='post')")
    W("        prob  = float(model.predict(X, verbose=0).flatten()[0])")
    W("        label = int(prob >= THRESHOLD)")
    W("    elif hasattr(model, 'predict_proba'):")
    W("        prob  = float(model.predict_proba(tfidf.transform([cleaned]))[0, 1])")
    W("        label = int(prob >= THRESHOLD)")
    W("    else:")
    W("        # LinearSVC / models without predict_proba:")
    W("        # decision_function scores are raw margins, NOT calibrated")
    W("        # probabilities. Sigmoid of a large margin is always ~1.0")
    W("        # which makes everything look like Disaster. Use predict()")
    W("        # directly — it gives the correct 0/1 label.")
    W("        label = int(model.predict(tfidf.transform([cleaned]))[0])")
    W("        prob  = float(label)")
    W("    conf = prob if label else 1.0 - prob")
    W("    return {")
    W("        'tweet':       text,")
    W("        'cleaned':     cleaned,")
    W("        'probability': round(prob, 4),")
    W("        'label':       'Disaster' if label else 'Non-Disaster',")
    W("        'is_disaster': bool(label),")
    W("        'confidence':  round(float(conf), 4),")
    W("    }")
    W()

    # -- session state --------------------------------------------------------
    W("if 'history' not in st.session_state:")
    W("    st.session_state.history = []")
    W()
    W("model, tfidf, lstm_tok, meta = load_predictor()")
    W()

    # -- sidebar --------------------------------------------------------------
    W("with st.sidebar:")
    W("    st.markdown('<div class=\"stag\">Model Info</div>', unsafe_allow_html=True)")
    W("    rows = [('Model', meta['best_model']), ('F1', meta['f1_score']),")
    W("            ('Accuracy', meta['accuracy']), ('Precision', meta['precision']),")
    W("            ('Recall', meta['recall']), ('Threshold', meta['best_threshold']),")
    W("            ('ROC-AUC', meta.get('roc_auc','N/A'))]")
    W("    h = ''.join(f'<div class=\"mrow\"><span class=\"mk\">{k}</span><span class=\"mv\">{v}</span></div>' for k,v in rows)")
    W("    st.markdown(h, unsafe_allow_html=True)")
    W("    st.divider()")
    W("    total = len(st.session_state.history)")
    W("    n_dis = sum(1 for x in st.session_state.history if x['is_disaster'])")
    W("    sh = (f'<div class=\"mrow\"><span class=\"mk\">Predictions</span><span class=\"mv\">{total}</span></div>'")
    W("          f'<div class=\"mrow\"><span class=\"mk\">Disaster</span><span class=\"mv\">{n_dis}</span></div>'")
    W("          f'<div class=\"mrow\"><span class=\"mk\">Safe</span><span class=\"mv\">{total-n_dis}</span></div>')")
    W("    st.markdown(sh, unsafe_allow_html=True)")
    W("    st.divider()")
    W("    if st.button('Clear History'):")
    W("        st.session_state.history = []")
    W("        st.rerun()")
    W()

    # -- header ---------------------------------------------------------------
    W("st.markdown('<h1 class=\"hero\">DisasterScan</h1>', unsafe_allow_html=True)")
    W("st.markdown('<p class=\"sub\">Real-time tweet disaster classification · NLP + Deep Learning</p>', unsafe_allow_html=True)")
    W("st.markdown('<br>', unsafe_allow_html=True)")
    W()

    # -- tabs -----------------------------------------------------------------
    W("tab1, tab2, tab3 = st.tabs(['Single Tweet', 'Batch CSV', 'History'])")
    W()

    # -- Tab 1 ----------------------------------------------------------------
    W("with tab1:")
    W("    st.markdown('#### Analyse a tweet')")
    W("    examples = ['Type your own...', 'BREAKING: Wildfire sweeps through California, thousands evacuated!',")
    W("                'Earthquake 6.2 strikes Japan coast. Tsunami warning issued.',")
    W("                'Flood waters rising in Houston - residents urged to evacuate.',")
    W("                'Just had the most amazing coffee this morning, loving life!',")
    W("                'I am on fire at the gym today, crushed my personal best!',")
    W("                'This film is a total disaster, worst thing I have seen all year.']")
    W("    sel = st.selectbox('Try an example or type below:', examples)")
    W("    default = '' if sel == examples[0] else sel")
    W("    tweet_input = st.text_area('Tweet', value=default, height=110,")
    W("                               placeholder='Paste or type a tweet here...',")
    W("                               label_visibility='collapsed')")
    W("    col_btn, col_info = st.columns([2, 5])")
    W("    with col_btn:")
    W("        go_btn = st.button('Analyse', use_container_width=True)")
    W("    with col_info:")
    W("        n = len(tweet_input)")
    W("        c = '#ff3c3c' if n > 280 else '#555577'")
    W("        st.markdown(f'<span style=\"font-size:.75rem;color:{c}\">{n} chars</span>', unsafe_allow_html=True)")
    W("    if go_btn:")
    W("        if not tweet_input.strip():")
    W("            st.warning('Please enter a tweet first.')")
    W("        else:")
    W("            with st.spinner('Scanning...'):")
    W("                res = run_predict(tweet_input, model, tfidf, lstm_tok, meta)")
    W("            st.session_state.history.insert(0, res)")
    W("            is_dis = res['is_disaster']")
    W("            cc  = 'dis' if is_dis else 'safe'")
    W("            lc  = 'lbl-dis' if is_dis else 'lbl-safe'")
    W("            ico = '[DISASTER]' if is_dis else '[SAFE]'")
    W("            conf_pct = round(res['confidence']*100, 1)")
    W("            thr = meta['best_threshold']")
    W("            card = (f'<div class=\"card {cc}\">'")
    W("                    f'<p class=\"lbl {lc}\">{ico} {res[\"label\"]}</p>'")
    W("                    f'<p class=\"meta\">Confidence: {conf_pct}% &nbsp;|&nbsp; Threshold: {thr}</p>'")
    W("                    f'<div class=\"badge\">Cleaned: {res[\"cleaned\"]}</div>'")
    W("                    f'</div>')")
    W("            st.markdown(card, unsafe_allow_html=True)")
    W("            cg, cb = st.columns([1, 2])")
    W("            with cg:")
    W("                st.plotly_chart(confidence_gauge(res['probability'], is_dis),")
    W("                                use_container_width=True, config={'displayModeBar': False})")
    W("            with cb:")
    W("                st.markdown('<br>', unsafe_allow_html=True)")
    W("                st.markdown('**Probability split**')")
    W("                st.plotly_chart(prob_bar(res['probability']),")
    W("                                use_container_width=True, config={'displayModeBar': False})")
    W("                dp = res['probability']")
    W("                st.markdown(f'<span class=\"pill\">disaster: {dp}</span>'")
    W("                            f'<span class=\"pill\">safe: {round(1-dp,4)}</span>',")
    W("                            unsafe_allow_html=True)")
    W()

    # -- Tab 2 ----------------------------------------------------------------
    W("with tab2:")
    W("    st.markdown('#### Batch prediction from CSV')")
    W("    st.caption('Upload a CSV with a text column. Add a target column (0/1) to see accuracy.')")
    W("    uploaded = st.file_uploader('Upload CSV', type='csv', label_visibility='collapsed')")
    W("    if uploaded:")
    W("        df = pd.read_csv(uploaded)")
    W("        if 'text' not in df.columns:")
    W("            st.error('CSV must have a text column.')")
    W("        else:")
    W("            st.info(f'{len(df)} tweets loaded')")
    W("            if st.button('Run Batch Prediction'):")
    W("                prog = st.progress(0)")
    W("                preds = []")
    W("                rows_list = df['text'].fillna('').tolist()")
    W("                for i, row in enumerate(rows_list):")
    W("                    preds.append(run_predict(row, model, tfidf, lstm_tok, meta))")
    W("                    prog.progress((i+1)/len(rows_list))")
    W("                prog.empty()")
    W("                out = pd.DataFrame(preds)[['tweet','label','probability','confidence','cleaned']]")
    W("                out.columns = ['Tweet','Prediction','Disaster Prob','Confidence','Cleaned']")
    W("                nt = len(out)")
    W("                nd = out['Prediction'].eq('Disaster').sum()")
    W("                c1, c2, c3 = st.columns(3)")
    W("                c1.metric('Total', nt)")
    W("                c2.metric('Disaster', nd)")
    W("                c3.metric('Safe', nt-nd)")
    W("                if 'target' in df.columns:")
    W("                    pl = [1 if p['is_disaster'] else 0 for p in preds]")
    W("                    acc = np.mean(np.array(df['target'].values) == np.array(pl))")
    W("                    st.success(f'Accuracy: {acc:.4f}')")
    W("                st.dataframe(out, use_container_width=True, height=320)")
    W("                st.download_button('Download CSV',")
    W("                    out.to_csv(index=False).encode('utf-8'),")
    W("                    'predictions.csv', 'text/csv')")
    W()

    # -- Tab 3 ----------------------------------------------------------------
    W("with tab3:")
    W("    st.markdown('#### Prediction history')")
    W("    if not st.session_state.history:")
    W("        st.info('No predictions yet. Use the Single Tweet tab first.')")
    W("    else:")
    W("        hdf = pd.DataFrame(st.session_state.history)[['tweet','label','probability','confidence']]")
    W("        hdf.columns = ['Tweet','Prediction','Disaster Prob','Confidence']")
    W("        st.dataframe(")
    W("            hdf.style.map(lambda v: 'color:#ff3c3c' if v=='Disaster' else 'color:#00e5a0',")
    W("                          subset=['Prediction']),")
    W("            use_container_width=True, height=400)")

print(f"app_path written: {_app_path}")
logger.info("app.py written to %s", _app_path)


app_path written: ..\deployment\app.py


---
### **SECTION 7 — Install Streamlit and Plotly (if not already installed)**

In [22]:
import importlib

for pkg in ['streamlit', 'plotly']:
    if importlib.util.find_spec(pkg) is None:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'{pkg} installed successfully')
    else:
        print(f'{pkg} already installed')

streamlit already installed
plotly already installed


---
### **SECTION 8 — Deployment Checklist**

In [23]:
logger.info('Running deployment checklist...')

checklist = [
    ('tfidf.pkl              (NB2)', MODEL_DIR / 'tfidf.pkl'),
    ('best_model.pkl         (NB2)', MODEL_DIR / 'best_model.pkl'),
    ('lstm_model.h5          (NB4)', MODEL_DIR / 'lstm_model.h5'),
    ('model_metadata.json    (NB5)', MODEL_DIR / 'model_metadata.json'),
    ('lstm_tokenizer.pkl     (NB5)', MODEL_DIR / 'lstm_tokenizer.pkl'),
    ('app.py                 (NB6)', DEPLOY_DIR / 'app.py'),
]

print(f"\n{'DEPLOYMENT CHECKLIST':=^55}")
all_pass = True
for label, path in checklist:
    ok       = Path(path).exists()
    all_pass = all_pass and ok
    print(f"{label}")
print('=' * 55)
if all_pass:
    print('All checks passed — ready to launch!')
else:
    print('Missing files above — re-run the relevant notebooks first.')
logger.info('Checklist complete. all_pass=%s', all_pass)


=================DEPLOYMENT CHECKLIST==================
tfidf.pkl              (NB2)
best_model.pkl         (NB2)
lstm_model.h5          (NB4)
model_metadata.json    (NB5)
lstm_tokenizer.pkl     (NB5)
app.py                 (NB6)
All checks passed — ready to launch!


---
### **SECTION 9 — Launch Streamlit App**

Starts the Streamlit server and **automatically opens `http://localhost:8501`** in your browser.

- **Success** — browser opens directly, URL and PID are printed
- **Already running** — browser opens to the existing session
- **Failure** — full Streamlit error output is shown with a list of common causes and fixes

> To stop the server: **Kernel > Interrupt** in Jupyter, or `Ctrl+C` in terminal.

In [24]:
import socket, subprocess, sys, time, webbrowser

APP_PATH = str((DEPLOY_DIR / 'app.py').resolve())
PORT     = 8501
URL      = f'http://localhost:{PORT}'


def port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0


# ── Already running ───────────────────────────────────────────────────────
if port_in_use(PORT):
    print(f'INFO: Streamlit is already running on port {PORT}.')
    print(f'      Opening {URL} in your browser...')
    webbrowser.open(URL)

# ── Start fresh ───────────────────────────────────────────────────────────
else:
    print(f'Starting DisasterScan...  (app: {APP_PATH})')

    proc = subprocess.Popen(
        [
            sys.executable, '-m', 'streamlit', 'run', APP_PATH,
            '--server.port',              str(PORT),
            '--server.headless',          'true',
            '--browser.gatherUsageStats', 'false',
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,   # captured separately for clean error reporting
        text=True,
    )

    # Poll until port is open (up to 20 seconds)
    ready = False
    for _ in range(40):
        time.sleep(0.5)
        if port_in_use(PORT):
            ready = True
            break

    # ── SUCCESS ───────────────────────────────────────────────────────────
    if ready:
        print()
        print('=' * 55)
        print('  DisasterScan launched successfully!')
        print(f'  URL  : {URL}')
        print(f'  PID  : {proc.pid}')
        print('  Stop : Kernel > Interrupt  (or Ctrl+C in terminal)')
        print('=' * 55)
        print()
        print(f'Opening {URL} in your browser...')
        webbrowser.open(URL)
        logger.info('Streamlit started on port %d  PID %d', PORT, proc.pid)

    # ── FAILURE ───────────────────────────────────────────────────────────
    else:
        try:
            stdout_out, stderr_out = proc.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            stdout_out, stderr_out = proc.communicate()

        print()
        print('=' * 55)
        print('  ERROR: Streamlit did not start within 20 seconds.')
        print('=' * 55)
        print()

        combined = (stdout_out or '') + (stderr_out or '')
        if combined.strip():
            print('--- Streamlit output ---')
            print(combined.strip())
            print('------------------------')
        else:
            print('(No output captured from Streamlit process.)')

        print()
        print('Common causes and fixes:')
        print('  1. streamlit not installed   ->  pip install streamlit plotly')
        print('  2. app.py has a syntax error ->  re-run Section 6 (Write app.py)')
        print('  3. Port already in use       ->  change PORT above to e.g. 8502')
        print('  4. Missing model files       ->  check Deployment Checklist (Section 8)')
        print()
        print(f'To run manually:')
        print(f'  streamlit run "{APP_PATH}" --server.port {PORT}')
        logger.error('Streamlit failed to start. stderr: %s', stderr_out)

logger.info('==== DEPLOYMENT PIPELINE COMPLETED ====')


INFO: Streamlit is already running on port 8501.
      Opening http://localhost:8501 in your browser...


---
### **Notebook 6 Summary**

| Section | Purpose |
|---|---|
| 1 | Load `model_metadata.json` from Notebook 5 |
| 2 | Load final model, TF-IDF, tokenizer — no rebuilding |
| 3 | `clean_text()` — same as NB1, required for self-contained export |
| 4 | `DisasterTweetPredictor` class |
| 5 | Smoke test — validates pipeline before writing the app |
| 6 | Write `requirements.txt` (Streamlit + Plotly) |
| 7 | Write `deployment/app.py` — full Streamlit UI |
| 8 | Auto-install Streamlit + Plotly if missing |
| 9 | Deployment checklist |
| 10 | **Launch** — `http://localhost:8501` opens in browser |

#### App features:
| Tab | What it does |
|---|---|
| ⚡ Single Tweet | Type or pick an example — shows verdict card, confidence gauge, probability split bar |
| 📋 Batch CSV | Upload CSV with `text` column — predictions, summary stats, optional accuracy, download |
| 🕒 History | Colour-coded table of every prediction made in the session |
| Sidebar | Model metrics (F1, AUC, etc.) + session stats + clear history |